In [215]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import math

In [193]:
data = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")

In [175]:
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 100)
data.columns = data.columns.str.lower()

In [151]:
data.info

<bound method DataFrame.info of       holiday    temp  rain_1h  snow_1h  clouds_all  weather_main  \
0         NaN  288.28      0.0      0.0          40        Clouds   
1         NaN  289.36      0.0      0.0          75        Clouds   
2         NaN  289.58      0.0      0.0          90        Clouds   
3         NaN  290.13      0.0      0.0          90        Clouds   
4         NaN  291.14      0.0      0.0          75        Clouds   
...       ...     ...      ...      ...         ...           ...   
48199     NaN  283.45      0.0      0.0          75        Clouds   
48200     NaN  282.76      0.0      0.0          90        Clouds   
48201     NaN  282.73      0.0      0.0          90  Thunderstorm   
48202     NaN  282.09      0.0      0.0          90        Clouds   
48203     NaN  282.12      0.0      0.0          90        Clouds   

          weather_description            date_time  traffic_volume  
0            scattered clouds  2012-10-02 09:00:00            5545  
1

In [196]:
print(data.columns)
print("----------------------------------")
print(data.shape)
print("----------------------------------")
print(data.isnull().sum()/len(data))

Index(['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main',
       'weather_description', 'date_time', 'traffic_volume'],
      dtype='object')
----------------------------------
(48204, 9)
----------------------------------
holiday                0.998735
temp                   0.000000
rain_1h                0.000000
snow_1h                0.000000
clouds_all             0.000000
weather_main           0.000000
weather_description    0.000000
date_time              0.000000
traffic_volume         0.000000
dtype: float64


In [197]:
profile = ProfileReport(data)
profile.to_file("df_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


%|          | 0/9 [00:00<?, ?it/s]
100%|██████████| 9/9 [00:00<00:00, 18.79it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [199]:
def preproccesing(data):
    data['holiday'] = data['holiday'].isnull().apply(lambda x: 0 if x else 1)
    data['rain_1h'] = (data['rain_1h'] > 0).astype(int)
    data['date_time'] = pd.to_datetime(data['date_time'])
    data['date'] = data['date_time'].dt.date
    data['hour'] = data['date_time'].dt.hour
    data['date'] = pd.to_datetime(data['date'])
    data['day_of_week'] = data['date'].dt.dayofweek
    data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)
    
    data = data.drop(columns = ['date_time'])
    data = data[data['temp'] > 100]
    data = data.drop(columns = ['snow_1h'])
    data = data.drop(columns = ['weather_description'])
    data = data.drop(columns = ['date'])
    
    return data
data = preproccesing(data)

In [202]:
print(data['weather_main'].value_counts())

weather_main
Clouds          15164
Clear           13381
Mist             5950
Rain             5672
Snow             2876
Drizzle          1821
Haze             1360
Thunderstorm     1034
Fog               912
Smoke              20
Squall              4
Name: count, dtype: int64


In [203]:
data.head()

,holiday,temp,rain_1h,clouds_all,weather_main,traffic_volume,hour,day_of_week,is_weekend
0,0,288.28,0,40,Clouds,5545,9,1,0
1,0,289.36,0,75,Clouds,4516,10,1,0
2,0,289.58,0,90,Clouds,4767,11,1,0
3,0,290.13,0,90,Clouds,5026,12,1,0
4,0,291.14,0,75,Clouds,4918,13,1,0


In [204]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [205]:
data = pd.get_dummies(data, drop_first=True, dtype=int)

In [206]:
X = data.drop(columns=['traffic_volume']) 
y = data['traffic_volume']

In [207]:
X_train, X_test, y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [211]:
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=8, random_state=42)
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [216]:
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=8, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred_xgb))
print("R2:", r2_score(y_test, y_pred_xgb))
print("Mean Error:",math.sqrt(mean_squared_error(y_test, y_pred_xgb)))

XGBoost MSE: 205202.078125
XGBoost R2: 0.9482450485229492
Mean Error: 452.9923598969413
